# 17 · New attack families: SSRF, XXE, SSTI, NoSQL, Log4Shell, LDAP, XPath

Until now the detector and the attack type knew four payload families (SQL injection, XSS, path traversal, command injection). This notebook adds seven more, so the system can name and explain them:

| Family | What the attacker sends |
|---|---|
| SSRF | a link to an internal address or the cloud metadata service, or a non-web scheme (gopher://, file://) |
| XXE | an XML document with a DOCTYPE / ENTITY that reads a file or a URL |
| SSTI | template code (`{{7*7}}`, `${...}`, `<% %>`) that the server runs |
| NoSQL | MongoDB operators (`user[$ne]=1`, `{"$gt": ""}`, `$where`) |
| Log4Shell | a Log4j lookup (`${jndi:ldap://...}`) |
| LDAP | LDAP filter syntax (`*)(uid=*))(|(uid=*`) |
| XPath | XPath syntax (`count(/child::node())`, `')] | //user/*`) |

**Data.** Payloads come from public lists (PayloadsAllTheThings, FuzzDB, SecLists and a Log4Shell bypass list, at fixed commits; see `ml/build_new_families.py`). Each payload is put inside a normal CSIC 2010 request (`ml/new_families.py`), like the HttpParams values. Per family, 30% of the payloads are held out: they are different strings, never used in training. The generator runs are used only to check that nothing else got worse. No locked file (run6, the sealed tool logs) is used.

**What changed in the code.**
- `ml/features.py`: one count per family (`v_kw_ssrf`, ..., matched on the decoded query and body, parameter names included, because NoSQL operators sit in the name), added to `v_kw_total`.
- `v_chr_brace` (any `{` or `}`) became `v_ctx_brace` (braces only where they open code). Section 4 shows why.
- `ml/attack_type.py`: 11 types; `ml/knowledge_base.py`: 8 new OWASP entries; `ml/defense.py`: WAF rules for the new families.

In [1]:
import sys
import warnings
from pathlib import Path
from urllib.parse import quote
import joblib
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
from backend.app.log_parser import decode, parse_log
from eval import benign_tricky, benign_tricky_new
from eval.final_test import run, summary, type_accuracy
from eval.rules import rule_predict
from ml import new_families
from ml.attack_type import NEW_TYPES, attack_types, load_type_model
from ml.defense import FixFinder, evidence_text, waf_rules
from ml.detector import MODEL_PATH, to_frame
from ml.train_defense import DEV_RUNS

D = ROOT / 'data'
detector, type_model = joblib.load(MODEL_PATH), load_type_model()
P = new_families.read_payloads()
pd.crosstab(P['family'], P['split'], margins=True)

split,heldout,train,All
family,,,
ldap,4,8,12
log4shell,9,21,30
nosql,14,33,47
ssrf,29,67,96
ssti,89,207,296
xpath,5,11,16
xxe,19,44,63
All,169,391,560


Payloads per family. LDAP and XPath have few public payloads, so their numbers below rest on a handful of strings; the report says so. Sources:

In [2]:
P.groupby(['family', 'source']).size().rename('payloads').to_frame()

payloads
family    source                                                      
ldap      PayloadsAllTheThings/LDAP Injection/Intruder/LD...         9
          PayloadsAllTheThings/LDAP Injection/Intruder/LD...         1
          PayloadsAllTheThings/LDAP Injection/README.md              1
          fuzzdb/attack/ldap/ldap-injection.txt                      1
log4shell PayloadsAllTheThings/CVE Exploits/Log4Shell.md            13
          log4j-bypass-words/README.md                              17
nosql     PayloadsAllTheThings/NoSQL Injection/Intruder/N...        24
          PayloadsAllTheThings/NoSQL Injection/README.md            21
          SecLists/Fuzzing/Databases/SQLi/NoSQL.txt                  2
ssrf      PayloadsAllTheThings/Server Side Request Forger...        34
          PayloadsAllTheThings/Server Side Request Forger...        11
          PayloadsAllTheThings/Server Side Request Forger...        51
ssti      PayloadsAllTheThings/Server Side Template Injec...         5
          PayloadsAllTheThings/Server Side Template Injec...       105
          PayloadsAllTheThings/Server Side Template Injec...        48
          PayloadsAllTheThings/Server Side Template Injec...        17
          PayloadsAllTheThings/Server Side Template Injec...        25
          PayloadsAllTheThings/Server Side Template Injec...        51
          PayloadsAllTheThings/Server Side Template Injec...         6
          PayloadsAllTheThings/Server Side Template Injec...         7
          SecLists/Fuzzing/template-engines-expression.txt           9
          SecLists/Fuzzing/template-engines-identificatio...        23
xpath     PayloadsAllTheThings/XPATH Injection/README.md             8
          fuzzdb/attack/xpath/xpath-injection.txt                    8
xxe       PayloadsAllTheThings/XXE Injection/Intruders/XX...        23
          PayloadsAllTheThings/XXE Injection/Intruders/xm...        30
          SecLists/Fuzzing/XXE-Fuzzing.txt                           4
          fuzzdb/attack/xml/xml-attacks.txt                          6

## 1. Detection: are the new families caught?

Recall on the requests that carry a payload (all of them are attacks). `rules` = the rules-only baseline (`eval/rules.py`, which now also has the new-family patterns); `model` = the retrained detector. The held-out part is the honest number: those payloads were never trained on.

In [3]:
def detection(split):
    X, _, fam, reqs = new_families.load(split)
    t = pd.DataFrame({'family': fam,
                      'rules': rule_predict([r[0] for r in reqs], [r[1] for r in reqs], [r[2] for r in reqs]),
                      'model': detector.predict(X)})
    out = t.groupby('family')[['rules', 'model']].mean()
    out.loc['ALL'] = t[['rules', 'model']].mean()
    out['requests'] = t.groupby('family').size().reindex(out.index).fillna(len(t)).astype(int)
    return out.round(3)

print('held out (never trained on)'); display(detection('heldout'))
print('trained on'); display(detection('train'))

held out (never trained on)


,rules,model,requests
family,,,
ldap,0.750,1.000,16
log4shell,1.000,1.000,36
nosql,0.929,0.929,56
ssrf,0.966,0.966,116
ssti,0.944,0.966,356
xpath,1.000,1.000,20
xxe,1.000,1.000,76
ALL,0.953,0.970,676


trained on


,rules,model,requests
family,,,
ldap,0.875,0.875,200
log4shell,1.000,1.000,210
nosql,0.970,0.970,231
ssrf,0.910,0.925,268
ssti,0.947,0.961,828
xpath,0.909,1.000,209
xxe,1.000,1.000,220
ALL,0.945,0.961,2166


Most misses are fragments that only work together with the server's answer (an SSTI engine probe such as `Dus{XXXXXXX}tjs`, a Razor `@(1+2)` whose `+` turns into a space after URL-decoding, a public link that redirects to an internal one). A log does not show the answer, so these stay a limitation.

## 2. Attack type on the held-out payloads

`rules` = pattern counts with the domain rules only; `rules + model` = the system (the type model breaks ties). XPath payloads such as `' or '1'='1` are also valid SQL injection: from the log alone they cannot be told apart, so most of them are called `sqli`.

In [4]:
X, _, fam, reqs = new_families.load('heldout')
m, p, b = [r[0] for r in reqs], [r[1] for r in reqs], [r[2] for r in reqs]
T = pd.DataFrame({'family': fam, 'rules': attack_types(m, p, b, None),
                  'rules + model': attack_types(m, p, b, type_model)})
acc = pd.DataFrame({c: (T[c] == T['family']).groupby(T['family']).mean() for c in ['rules', 'rules + model']})
acc.loc['ALL'] = [(T[c] == T['family']).mean() for c in ['rules', 'rules + model']]
display(acc.round(3))
pd.crosstab(T['family'], T['rules + model'], rownames=['true'], colnames=['predicted'])

,rules,rules + model
family,,
ldap,0.750,0.812
log4shell,1.000,1.000
nosql,0.929,0.929
ssrf,0.966,0.991
ssti,0.921,0.955
xpath,0.400,0.400
xxe,1.000,1.000
ALL,0.923,0.947


predicted,cmdi,ldap,log4shell,nosql,sqli,ssrf,ssti,xpath,xxe
true,,,,,,,,,
ldap,0,13,0,0,0,0,3,0,0
log4shell,0,0,36,0,0,0,0,0,0
nosql,0,0,0,52,0,2,2,0,0
ssrf,0,0,0,1,0,115,0,0,0
ssti,8,5,0,0,0,3,340,0,0
xpath,0,0,0,0,12,0,0,8,0
xxe,0,0,0,0,0,0,0,0,76


## 3. Nothing else got worse: development runs

The full pipeline (`eval/final_test.run`: detector + session layer + attack type) on the five dev runs. For comparison, the same runs on `devlope` 04ec655 (before this change) gave model + sessions recall 0.998 / 1.000 / 0.993 / 0.990 / 0.983, the same false alarms (0 / 0 / 0 / 30 / 164; the camp7 ones are normal pages inside attacker sessions, see notebook 11) and type accuracy 0.958 / 0.958 / 0.959 / 0.968 / 0.948.

In [5]:
rows = []
for log, labels in DEV_RUNS:
    req, sessions, report = run(D / 'raw' / log, D / 'labels' / labels)
    s = summary(req)
    rows.append({'run': log.replace('_generated_', ' ').replace('.log', ''),
                 'model + sessions recall': s.loc['model + sessions', 'recall'],
                 'false alarms': s.loc['model + sessions', 'false alarms'],
                 'rules + sessions recall': s.loc['rules + sessions', 'recall'],
                 'model alone false alarms': s.loc['model', 'false alarms'],
                 'type accuracy': round(type_accuracy(req)[1], 3)})
pd.DataFrame(rows).set_index('run')

,model + sessions recall,false alarms,rules + sessions recall,model alone false alarms,type accuracy
run,,,,,
juiceshop run3,0.999,0,0.985,0,0.958
juiceshop run4,1.000,0,0.996,0,0.958
dvwa run4,0.993,0,0.983,0,0.959
juiceshop camp7,0.992,30,0.976,0,0.968
dvwa camp7,0.984,164,0.965,0,0.945


## 4. False alarms on hard normal text

`eval/benign_tricky.py` is the old set (SQL words in shop comments). `eval/benign_tricky_new.py` was written for this change: normal text with links, braces, dollar signs and parentheses. The first try of this change, and the detector before it, flagged 12 of its 60 requests: any `{` or `}` in a value counted as an attack sign (`v_chr_brace`), because in CSIC no normal value has braces. `v_ctx_brace` counts braces only where they open code (`{{7*7}}`, `#{1+1}`, `{"$ne": 1}`), and `Dear {customer}` is no longer flagged.

In [6]:
for name, mod in [('old hard set', benign_tricky), ('new hard set', benign_tricky_new)]:
    r = mod.rows()
    pred = detector.predict(to_frame(r['method'], r['path'], r['body']))
    print(f'{name}: {int(pred.sum())} of {len(r)} flagged')
    for i in pred.nonzero()[0]:
        print('   ', r['path'][i] if r['method'][i] == 'GET' else r['body'][i])

old hard set: 0 of 60 flagged
new hard set: 4 of 60 flagged
    /rest/products/search?q=use%20code%20${SAVE10}%20at%20checkout
    {"comment": "use code ${SAVE10} at checkout", "rating": 5}
    /rest/products/search?q=a%20||%20b,%20both%20fine
    {"comment": "a || b, both fine", "rating": 5}


The 4 left are shell syntax: `${SAVE10}` and `a || b`. Both are exactly what command injection uses (`${IFS}`, `|| ping`), so we keep them as attack signs and list them as a known limitation.

## 5. HttpParams test split and the public log

HttpParams test (old families, never trained on). Before this change: recall 0.992, 1 false alarm. The 40 extra misses are 39 fuzz-list templates with a `{file}` placeholder (`/....{file}`) that the old model caught only because of the braces, and 1 XSS string.

In [7]:
hp = pd.read_csv(D / 'raw' / 'httpparams_test.csv', keep_default_na=False)
Xh = to_frame(['GET'] * len(hp), ['/?q=' + quote(v, safe='') for v in hp['payload']], [''] * len(hp))
ph, yh = detector.predict(Xh), (hp['label'] == 'anom').values
print('recall', round(ph[yh].mean(), 4), '| false alarms', int(ph[~yh].sum()), 'of', int((~yh).sum()))
missed = hp[yh & (ph == 0)]
print('missed:', len(missed), '| with a {file} placeholder:', int(missed['payload'].str.contains('{file}', regex=False).sum()))

recall 0.9819 | false alarms 1 of 6434
missed: 71 | with a {file} placeholder: 39


The public Apache log (semicomplete.com, 10,000 real requests, no labels; `data/raw/public_apache_logs.log`, optional). Before this change the detector alone flagged 66 requests there (WordPress probes and downloads; `served_files` in the pipeline clears the downloads). The new families must add none.

In [8]:
pub = D / 'raw' / 'public_apache_logs.log'
if pub.exists():
    rows, skipped = parse_log(decode(pub.read_bytes()))
    Xp = to_frame([r['method'] for r in rows], [r['path'] for r in rows], [r['body'] for r in rows])
    pp = detector.predict(Xp)
    fams = [c for c in Xp.columns if c in ('v_kw_ssrf', 'v_kw_xxe', 'v_kw_ssti', 'v_kw_nosql', 'v_kw_log4j', 'v_kw_ldap', 'v_kw_xpath')]
    print(len(rows), 'requests,', int(pp.sum()), 'flagged; with a new-family sign:', int((Xp[fams].sum(axis=1) > 0).sum()))
else:
    print('public log not found: skipped')

9999 requests, 66 flagged; with a new-family sign: 0


## 6. Fix and WAF rule for each new family

One held-out request per family through the k-NN (`FixFinder`) and the WAF rule builder.

In [9]:
finder = FixFinder()
held = pd.DataFrame(reqs, columns=['method', 'path', 'body', 'family'])
for fam in NEW_TYPES:
    one = held[held['family'] == fam].head(1).assign(status=200, timestamp='2026-10-30T10:00:00Z')
    entry, sim = finder.nearest(evidence_text(fam, one), k=1, family=fam)[0]
    rule = [r for r in waf_rules('203.0.113.7', 'allow', {fam: one}) if r.startswith('SecRule')]
    print(f'{fam:10s} -> {entry["id"]:15s} {entry["cwe"]:9s} {entry["title"]}')
    print('            ', entry['fix'][0])
    print('            ', rule[0].splitlines()[-1].strip()[:150] if rule else '-')

ssrf       -> ssrf-internal   CWE-918   Server-side request forgery to an internal address
             Do not fetch a URL the user sends; if the feature needs it, allow only a fixed list of hosts and schemes (https to named partner domains).
             SecRule ARGS:id|ARGS:nombre|ARGS:precio "@rx (?i)(?:gopher|dict|file|ldap|tftp)://|(?:https?|ftp)://(?:[^/?#@]*@)?(?:localhost|127\.|0\.0\.0\.0|\[|169
xxe        -> xxe-entity      CWE-611   XML external entity (XXE) in an XML upload or API
             Turn off DTDs (DOCTYPE) completely in the XML parser; this blocks XXE and the entity-expansion denial of service.
             SecRule REQUEST_BODY "@rx (?i)<!(?:doctype|entity)|xi:include" "t:none,t:urlDecodeUni,t:urlDecodeUni,t:removeComments"
ssti       -> ssti-template   CWE-1336  Server-side template injection
             Never build a template from user input; pass user input to a fixed template as data.
             SecRule ARGS:modo|ARGS:login|ARGS:password "@rx \{\{.*?\}\}|\{

## 7. Limitations (for the report)

- **Few payloads for some families.** LDAP (12) and XPath (16) public payloads; their held-out numbers rest on 4-5 strings.
- **Synthetic context.** Payloads sit inside CSIC requests, not real traffic of a site with that weakness. The locked final test (generator v4 with these families, run6) is the real check.
- **XPath vs SQL injection.** Many XPath payloads are also SQL injection strings; the type then says `sqli`.
- **Log4Shell in headers.** Real Log4Shell attacks are often in the User-Agent or other headers. The detector reads the URL and the body; an access log usually has no other headers.
- **Next step (Markov).** It knows only the old steps, because the campaign runs have none of the new families.
- **Shell syntax in prose.** `${...}` and `||` in normal text are still flagged (section 4).